In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text catalogo default "catalog_au";
create widget text esquema_source default "silver";
create widget text esquema_sink default "golden";

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df_silver = spark.table(f"{catalogo}.{esquema_source}.conceptos_calidad_trazabilidad")

In [0]:
df_silver_clean = df_silver.fillna({
    "nombre_del_dominio": "No Especificado",
    "nombre_del_subdominio": "No Especificado",
    "prioridad_del_entidad_de_dato": "Media",
    "dato_critico": "No",
    "personal": "No",
    "sensible": "No"
})

# Estandarizar valores SI / NO
df_silver_clean = df_silver_clean.withColumn(
    "dato_critico", 
    when(trim(col("dato_critico")).isin("Sí", "SI", "si", "sí", "S"), "Sí").otherwise("No")
).withColumn(
    "personal", 
    when(trim(col("personal")).isin("Sí", "SI", "si", "sí", "S"), "Sí").otherwise("No")
).withColumn(
    "sensible", 
    when(trim(col("sensible")).isin("Sí", "SI", "si", "sí", "S"), "Sí").otherwise("No")
)

In [0]:
df_reporte_calidad = df_silver_clean.groupBy("nombre_del_dominio", "nombre_del_subdominio").agg(
    countDistinct("codigo_de_entidad_del_dato").alias("total_conceptos"),
    countDistinct(when(col("dato_critico") == "Sí", col("codigo_de_entidad_del_dato"))).alias("total_datos_criticos"),
    countDistinct(when(col("personal") == "Sí", col("codigo_de_entidad_del_dato"))).alias("total_datos_personales"),
    countDistinct(when(col("sensible") == "Sí", col("codigo_de_entidad_del_dato"))).alias("total_datos_sensibles"),
    count(col("id_regla_de_calidad")).alias("total_reglas_calidad"),
    coalesce(round(avg(col("umbral_superior")), 4), lit(0.0)).alias("promedio_umbral_superior"),
    coalesce(round(avg(col("umbral_inferior")), 4), lit(0.0)).alias("promedio_umbral_inferior")
).withColumn("ultimo_procesamiento", current_timestamp())

In [0]:
df_resumen_prioridad = df_silver_clean.groupBy("nombre_del_dominio", "prioridad_del_entidad_de_dato").agg(
    countDistinct("codigo_de_entidad_del_dato").alias("cantidad_conceptos")
).withColumn("ultimo_procesamiento", current_timestamp())

In [0]:
target_table_1 = f"{catalogo}.{esquema_sink}.reporte_calidad_conceptos"
print(f"Escribiendo en la tabla Golden 1: {target_table_1}...")
df_reporte_calidad.write.mode("overwrite").insertInto(target_table_1)

In [0]:
target_table_2 = f"{catalogo}.{esquema_sink}.resumen_prioridad_dominio"
print(f"Escribiendo en la tabla Golden 2: {target_table_2}...")
df_resumen_prioridad.write.mode("overwrite").insertInto(target_table_2)